## Error in Horizontal flip V2 and Vertical Flip V2 functions:




In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
## General Imports
import numpy as np
#import cv2
from numpy import asarray
import  scipy.ndimage as sn
import math
import scipy.io
from collections import deque

## Tensorflow imports
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Layer,Dense,LayerNormalization,Input,Flatten,Softmax
from tensorflow.keras.layers import Concatenate,Multiply,Conv3D,LeakyReLU,Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
## Sparse Pruning
#import tensorflow_model_optimization as tfmot

##Others
from sklearn.decomposition import PCA
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, cohen_kappa_score

In [ ]:
def PCA_fit_transform(XData,ncmp):
    pca_obj=PCA(n_components=ncmp)
    xdata_shape=XData.shape
    XData=XData.reshape(-1,xdata_shape[-1])
    pca_obj.fit(XData)
    XData=pca_obj.transform(XData)
    XData=XData.reshape(xdata_shape[0],xdata_shape[1],xdata_shape[2],XData.shape[-1])
    return XData,pca_obj
## Shufling
def unison_shuffled_copies(a, b, c):
    assert len(a) == len(b) == len(c)
    p = np.random.permutation(len(a))
    return a[p], b[p], c[p]
## Agumenting Data
def horizontal_flipping(XTrain):
    XHTrain=np.zeros((XTrain.shape))
    for i in range(len(XTrain)):
        for bn in range(XTrain.shape[-1]):
            XHTrain[i,:,:,bn]=np.fliplr(np.copy(XTrain[i,:,:,bn]))
    return XHTrain
def vertical_flipping(XTrain):
    XHTrain=np.zeros((XTrain.shape))
    for i in range(len(XTrain)):
        for bn in range(XTrain.shape[-1]):
            XHTrain[i,:,:,bn]=np.flipud(np.copy(XTrain[i,:,:,bn]))
    return XHTrain


def cal_angle(Vec1,Vec2):
    mag1=np.linalg.norm(Vec1)
    mag2=np.linalg.norm(Vec2)
    dot_product=np.dot(Vec1,Vec2)
    if(mag1==0 or mag2==0):
        return 0
    cos_theta=dot_product/(mag1*mag2)
    cos_theta= -1 if(cos_theta <-1) else cos_theta
    cos_theta= 1 if(cos_theta>1) else cos_theta
    spec_angle=np.arccos(cos_theta)
    spec_angle=np.degrees(spec_angle)
    return spec_angle



def cal_SAM_mas_Xformer(XTrain):
  ##Cosine similarity
    max_angle=0
    ip_shape=XTrain.shape
    patch_width=ip_shape[2]
    mx_dis=(patch_width-1)/2
    SAM_masks=np.zeros((ip_shape[0],patch_width,patch_width))
    for smpl in range(ip_shape[0]):
        cp_pos=math.floor((patch_width/2))
        cntr_px=XTrain[smpl,cp_pos,cp_pos,:]
        mag1=np.linalg.norm(cntr_px)
        for rw in range(patch_width):
            for cl in range(patch_width):
                if(rw==cp_pos and cl==cp_pos):
                    SAM_masks[smpl,rw,cl]=1
                    continue
                curr_px=XTrain[smpl,rw,cl,:]
                mag2=np.linalg.norm(curr_px)
                chb_dist=max(abs(mx_dis-rw),abs(mx_dis-cl))
                wt_dist=(mx_dis+1-chb_dist)/(mx_dis+1)
                # mask_wgt=wt_dist
                if(mag1==0 or mag2==0):
                    SAM_masks[smpl,rw,cl]=wt_dist/2
                    continue
                cos_sim=np.dot(cntr_px,curr_px)/(mag1*mag2)
                cos_sim=0 if(cos_sim<0) else cos_sim
                cos_sim=1 if(cos_sim>1) else cos_sim
                mask_wgt=(cos_sim+wt_dist)/2
                SAM_masks[smpl,rw,cl]=mask_wgt
    #print('Max Angle : ', max_angle)
    return SAM_masks



def is_Test_Patch_Not_Overlaps(X_Train_Patch_Centres,X_Test_Patch_Centres,patch_size):
    extra_width=patch_size//2
    ITPNO=np.ones((len(X_Test_Patch_Centres)),dtype=np.int32)
    for i in range(len(X_Test_Patch_Centres)):
        for j in range(len(X_Train_Patch_Centres)):
            ## function for chessboard distance between two patch centre
            chb=max(abs(X_Test_Patch_Centres[i][0]-X_Train_Patch_Centres[j][0]),abs(X_Test_Patch_Centres[i][1]-X_Train_Patch_Centres[j][1]))
            if(chb <= (2*extra_width)):
                ITPNO[i]=0
                break
    print('Total Test Patches: ', len(ITPNO))
    print('Total Test Patches That dont Overlaps: ', np.sum(ITPNO))
    return ITPNO


def dataset_preparation(patch_size):
    extra_width=patch_size//2
    temp_pad=np.pad(temp,((extra_width,extra_width),(extra_width,extra_width),(0,0)),mode='edge')
    gt_temp_pad=np.pad(gt_temp,((extra_width,extra_width),(extra_width,extra_width)),mode='edge')
    print(temp_pad.shape,temp.shape,gt_temp_pad.shape)
    ## Patches Extracting
    i_h=temp_pad.shape[0]
    i_w=temp_pad.shape[1]
    X_Images=list()
    Y_Labels=list()
    X_Patch_Centres=list()
    for hi in range(extra_width,i_h-extra_width):
        for wi in range(extra_width,i_w-extra_width):
            if(gt_temp_pad[hi][wi]==0):
                continue
            h_start=hi-extra_width
            w_start=wi-extra_width
            h_end=hi+extra_width
            w_end=wi+extra_width
            mini_patch=temp_pad[h_start:h_end+1,w_start:w_end+1,:]
            mini_class=gt_temp_pad[hi][wi]
            #Patch Centres
            mini_patch_centre=np.zeros((2),dtype=np.int32)
            mini_patch_centre[0]=hi
            mini_patch_centre[1]=wi
            #
            X_Images.append(mini_patch)
            Y_Labels.append(mini_class)
            X_Patch_Centres.append(mini_patch_centre)
    ##
    X_Images=np.asarray(X_Images)
    Y_Labels=np.asarray(Y_Labels)
    X_Patch_Centres=np.asarray(X_Patch_Centres)
    print(X_Images.shape,Y_Labels.shape,X_Patch_Centres.shape)
    ## Shuffling
    X_Images,Y_Labels,X_Patch_Centres=unison_shuffled_copies(X_Images,Y_Labels,X_Patch_Centres)
    ##
    # Trn_IP=[0,10,71,41,11,24,36,10,23,10,48,122,29,10,63,19,10]
    Trn_IP_counter=[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
    X_Train=list()
    Y_Train=list()
    X_Train_Patch_Centres=list()
    X_Test=list()
    Y_Test=list()
    X_Test_Patch_Centres=list()
    ##
    for i in range(len(X_Images)):
        cls_i=Y_Labels[i]
        if(Trn_IP_counter[cls_i]<Trn_IP[cls_i]):
            X_Train.append(X_Images[i])
            Y_Train.append(Y_Labels[i])
            X_Train_Patch_Centres.append(X_Patch_Centres[i])
            Trn_IP_counter[cls_i]=Trn_IP_counter[cls_i]+1
        else:
            X_Test.append(X_Images[i])
            Y_Test.append(Y_Labels[i])
            X_Test_Patch_Centres.append(X_Patch_Centres[i])
    ##Free Up Ram
    del X_Images
    del Y_Labels
    del X_Patch_Centres
    del gt_temp_pad
    del temp_pad
    ##
    X_Train=np.asarray(X_Train)
    Y_Train=np.asarray(Y_Train)
    X_Train_Patch_Centres=np.asarray(X_Train_Patch_Centres)
    X_Test=np.asarray(X_Test)
    Y_Test=np.asarray(Y_Test)
    X_Test_Patch_Centres=np.asarray(X_Test_Patch_Centres)
    ##
    print('priting after dividing all samples into train and test')
    print(X_Train.shape,Y_Train.shape,Y_Test.shape,X_Test.shape)
    print('Patch_Centres shape Train and Test',X_Train_Patch_Centres.shape,X_Test_Patch_Centres.shape)
    ## Validation dataset preparation
    # Vld_IP=[0,10,71,41,11,24,36,10,23,5,48,122,29,10,63,19,10]
    Vld_IP_counter=[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]
    X_Vld=list()
    Y_Vld=list()
    X_Vld_Patch_Centres=list()
    New_X_Test=list()
    New_Y_Test=list()
    New_X_Test_Patch_Centres=list()
    for i in range(len(X_Test)):
        cls_i=Y_Test[i]
        if(Vld_IP_counter[cls_i]<Vld_IP[cls_i]):
            X_Vld.append(X_Test[i])
            Y_Vld.append(Y_Test[i])
            X_Vld_Patch_Centres.append(X_Test_Patch_Centres[i])
            Vld_IP_counter[cls_i]=Vld_IP_counter[cls_i]+1
        else:
            New_X_Test.append(X_Test[i])
            New_Y_Test.append(Y_Test[i])
            New_X_Test_Patch_Centres.append(X_Test_Patch_Centres[i])
    ##Free Up Ram
    del X_Test
    del Y_Test
    X_Test=np.asarray(New_X_Test)
    Y_Test=np.asarray(New_Y_Test)
    X_Test_Patch_Centres=np.asarray(New_X_Test_Patch_Centres)
    X_Vld=np.asarray(X_Vld)
    Y_Vld=np.asarray(Y_Vld)
    X_Vld_Patch_Centres=np.asarray(X_Vld_Patch_Centres)
    ##
    del New_X_Test,New_Y_Test,New_X_Test_Patch_Centres
    print('Printing after divinding Test into Validation and Test')
    print(X_Train.shape,X_Vld.shape,X_Test.shape)
    print(Y_Train.shape,Y_Vld.shape,Y_Test.shape)
    print(X_Train_Patch_Centres.shape,X_Vld_Patch_Centres.shape,X_Test_Patch_Centres.shape)
    # Is Test patch overlaps with train patches
    ITPNO=is_Test_Patch_Not_Overlaps(X_Train_Patch_Centres,X_Test_Patch_Centres,patch_size)
    #
    list_class_length=list()
    for i in range(17):
        x=np.sum(Y_Train==i)
        y=np.sum(Y_Test==i)
        z=np.sum(Y_Vld==i)
        list_class_length.append((x+y+z))
        print(x,z,y)
    print(list_class_length)

    #Agumentation
    X_Train,Y_Train=data_agumentation(X_Train,Y_Train)
    #
    X_og_Tr=cal_SAM_mas_Xformer(X_Train)
    X_og_Ts=cal_SAM_mas_Xformer(X_Test)
    X_og_Vld=cal_SAM_mas_Xformer(X_Vld)
    #
    permutation = np.random.permutation(len(X_Train))
    X_Train=X_Train[permutation]
    Y_Train=Y_Train[permutation]
    X_og_Tr=X_og_Tr[permutation]
    #
    print(X_Train.shape,Y_Train.shape)
    print(X_Vld.shape,Y_Vld.shape)
    print(X_Test.shape,Y_Test.shape)
    print(X_og_Tr.shape,X_og_Vld.shape,X_og_Ts.shape)
    #hot encoding
    ## Training Datsets
    Y_Tr_hot=Y_Train-1
    Y_Tr_hot=to_categorical(Y_Tr_hot)
    ## Validation Dataset
    Y_Vld_hot=Y_Vld-1
    Y_Vld_hot=to_categorical(Y_Vld_hot)
    ## Testing Data
    #labels
    Y_Ts_hot=Y_Test-1
    Y_Ts_hot=to_categorical(Y_Ts_hot)
    return X_Train,Y_Train,Y_Tr_hot,X_Test,Y_Test,Y_Ts_hot,X_Vld,Y_Vld,Y_Vld_hot,X_og_Tr,X_og_Vld,X_og_Ts,ITPNO

def data_agumentation(X_Train,Y_Train):
    #
    X_Train_Copy2=np.copy(X_Train)
    Y_Train_Copy2=np.copy(Y_Train)
    #
    X_hf_Train=horizontal_flipping(X_Train_Copy2)
    print(X_hf_Train.shape)
    X_Train=np.concatenate((X_Train,X_hf_Train),axis=0)
    del X_hf_Train
    #
    X_vf_Train=vertical_flipping(X_Train_Copy2)
    print(X_vf_Train.shape)
    X_Train=np.concatenate((X_Train,X_vf_Train),axis=0)
    del X_vf_Train
    #
    X_r90_Train=np.rot90(np.copy(X_Train_Copy2),k=1,axes=(1,2))
    print(X_r90_Train.shape)
    X_Train=np.concatenate((X_Train,X_r90_Train),axis=0)
    del X_r90_Train
    #
    X_r180_Train=np.rot90(np.copy(X_Train_Copy2),k=2,axes=(1,2))
    print(X_r180_Train.shape)
    X_Train=np.concatenate((X_Train,X_r180_Train),axis=0)
    del X_r180_Train
    #
    X_r270_Train=np.rot90(np.copy(X_Train_Copy2),k=3,axes=(1,2))
    print(X_r270_Train.shape)
    X_Train=np.concatenate((X_Train,X_r270_Train),axis=0)
    del X_r270_Train

    #
    Y_Train=np.concatenate((Y_Train,Y_Train_Copy2),axis=0)
    Y_Train=np.concatenate((Y_Train,Y_Train_Copy2),axis=0)
    Y_Train=np.concatenate((Y_Train,Y_Train_Copy2),axis=0)
    Y_Train=np.concatenate((Y_Train,Y_Train_Copy2),axis=0)
    Y_Train=np.concatenate((Y_Train,Y_Train_Copy2),axis=0)

    #deletions
    del Y_Train_Copy2,X_Train_Copy2
    #return
    return X_Train,Y_Train



def hot_to_labels(Y_hot):
    Y_labels=np.zeros((len(Y_hot),1))
    for smpl in range(len(Y_hot)):
        pred_label=np.argmax(Y_hot[smpl])+1
        Y_labels[smpl]=pred_label
    return Y_labels

def hot_to_labels_2(Y_hot):
    Y_labels=np.zeros((len(Y_hot),1))
    for smpl in range(len(Y_hot)):
        pred_label=np.argmax(Y_hot[smpl])+1
        print(pred_label)
        Y_labels[smpl][0]=pred_label
    return Y_labels

def cal_class_accuracies(Y_True,Y_Pred,num_classes):
    cc_true=np.zeros((num_classes,1))
    cc_correct=np.zeros((num_classes,1))
    count=0;
    for smpl in range(len(Y_True)):
        lbl=Y_True[smpl]
        if(Y_Pred[smpl]==lbl):
            cc_correct[lbl-1] +=1
            count +=1
        cc_true[lbl-1] +=1
    cls_accuracy=cc_correct/cc_true
    print('Class accuracies')
    print(cls_accuracy)
    print('Avergae accuracy')
    print(np.mean(cls_accuracy))
    print('Overall accuracy')
    print(count/len(Y_True))
    return cls_accuracy,np.mean(cls_accuracy),(count/len(Y_True))



In [ ]:
def total_hsimage_prediction(patch_size,modelhsi):
    count=0
    patch_predictions=np.zeros(gt_temp.shape)
    extra_width=patch_size//2
    ## Padding
    temp_pad=np.pad(temp,((extra_width,extra_width),(extra_width,extra_width),(0,0)),mode='edge')
    gt_temp_pad=np.pad(gt_temp,((extra_width,extra_width),(extra_width,extra_width)),mode='edge')
    print('Temp pad shape : ',temp_pad.shape,' GT pad shape : ',gt_temp_pad.shape)
    print('temp shape : ',temp.shape,' GT shape : ',gt_temp.shape)
    ## Patches Extracting
    i_h=temp_pad.shape[0]
    i_w=temp_pad.shape[1]
    ##
    total_samples=int(np.sum(gt_temp>0))
    print('Total samples to predict : ',total_samples)
    label_positions=np.zeros((total_samples,2),dtype=int)
    X_all=np.zeros((total_samples,patch_size,patch_size,temp.shape[-1]))
    sample_count=0

    for hi in range(extra_width,i_h-extra_width):
        for wi in range(extra_width,i_w-extra_width):
            mini_class=gt_temp_pad[hi][wi]
            if(mini_class==0):
                continue
            h_start=hi-extra_width
            w_start=wi-extra_width
            h_end=hi+extra_width
            w_end=wi+extra_width
            #
            mini_patch=temp_pad[h_start:h_end+1,w_start:w_end+1,:]
            X_all[sample_count,:,:,:]=mini_patch
            #
            label_positions[sample_count][0]=h_start
            label_positions[sample_count][1]=w_start
            sample_count=sample_count+1

    X2_all=cal_SAM_mas_Xformer(X_all)
    X2_all=np.expand_dims(X2_all,axis=(-2,-1))
    X_all=np.expand_dims(X_all,axis=(-1))
    print('Input1 shape: ',X_all.shape)
    print('Input2 shape: ',X2_all.shape)
    Y_pred_hot=modelhsi.predict([X_all,X2_all],batch_size=1024,verbose=0)
    Y_pred=hot_to_labels(Y_pred_hot)
    print(f"total sample = {total_samples}, len of pridiction = {len(Y_pred)}")
    for i in range(total_samples):
      x_pos=label_positions[i][0]
      y_pos=label_positions[i][1]
      patch_predictions[x_pos][y_pos]=Y_pred[i].item()
      # if(patch_predictions[x_pos][y_pos]==gt_temp[x_pos][y_pos]):
      #   count=count+1
    print('Correct count : ',int( np.sum(patch_predictions==gt_temp)-np.sum(gt_temp==0) ) )
    return patch_predictions

In [ ]:
def gelu_approx(x):
    return 0.5 * x * (1.0 + tf.tanh(tf.sqrt(2.0 / 3.141592653589793) *
                                     (x + 0.044715 * tf.pow(x, 3))))



class LearnableMultiplication(Layer):
    def __init__(self, input_dim, output_dim):
        super(LearnableMultiplication, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.kernel_regularizer=rl_l1_l2
    def build(self, input_shape):
        # Define a learnable weight matrix of shape (input_dim, output_dim)
        self.B = self.add_weight(
            shape=(self.input_dim, self.output_dim),
            initializer='random_normal',
            trainable=True,
            name='learnable_matrix',
            regularizer=self.kernel_regularizer
        )

    def call(self, inputs):
        # Perform tensordot on the last axis of inputs and first axis of B
        return tf.tensordot(inputs, self.B, axes=[-1, 0])  # result: (..., output_dim)



class MultiHeadAttention(Layer):
    def __init__(self,input_dim,num_heads=5):
        super(MultiHeadAttention, self).__init__()
        # assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        # self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.input_dim=input_dim
        self.head_dim =max(input_dim//num_heads,5)
        self.key_dim=max(input_dim//num_heads,5)
        self.key_embed_dim=self.key_dim * num_heads
        self.Softmax=Softmax(axis=-1)
        self.leaklyReLU=LeakyReLU(alpha=0.1)

        # Projections
        self.q_proj = LearnableMultiplication(input_dim=self.input_dim, output_dim=self.key_embed_dim)
        self.k_proj = LearnableMultiplication(input_dim=self.input_dim, output_dim=self.key_embed_dim)
        self.v_proj = LearnableMultiplication(input_dim=self.input_dim, output_dim=self.key_embed_dim)

        # Final output projection
        self.output_proj = LearnableMultiplication(input_dim=self.key_embed_dim, output_dim=self.input_dim)

    def split_heads(self, x):
        # x: (B, H, W, P, key_embed_dim)
        B, H, W, P, D = tf.unstack(tf.shape(x))
        #print(D//self.num_heads)
        x = tf.reshape(x, (B, H, W, P, self.num_heads, D//self.num_heads))
        return tf.transpose(x, perm=[0, 1, 2, 4, 3, 5])  # (B, H, W, heads, P, head_dim)

    def combine_heads(self, x):
        # x: (B, H, W, heads, P, head_dim)
        x = tf.transpose(x, perm=[0, 1, 2, 4, 3, 5])  # (B, H, W, P, heads, head_dim)
        B, H, W, P,c_heads,c_head_dim = tf.unstack(tf.shape(x))
        return tf.reshape(x, (B, H, W, P,c_heads*c_head_dim ))

    def call(self, inputs):
        # inputs: (B, H, W, P, 5)
        Q = self.q_proj(inputs)
        K = self.k_proj(inputs)
        V = self.v_proj(inputs)

        Q = self.split_heads(Q)  # (B, H, W, heads, P, head_dim)
        K = self.split_heads(K)
        V = self.split_heads(V)

        dk = tf.cast(self.key_dim, tf.float32)

        # Attention: (B, H, W, heads, P, P)
        scores = tf.matmul(Q, K, transpose_b=True) / tf.math.sqrt(dk)
        B_s,H_s,W_s,Hd_s,P1_s,P2_s=tf.unstack(tf.shape(scores))
        scores= tf.reshape(scores,(B_s,H_s*W_s,Hd_s,P1_s,P2_s))
        attn_weights = self.Softmax(scores)
        attn_weights=tf.reshape(attn_weights,(B_s,H_s,W_s,Hd_s,P1_s,P2_s))

        # Apply attention: (B, H, W, heads, P, head_dim)
        attended = tf.matmul(attn_weights, V)

        # Combine heads and project
        concat = self.combine_heads(attended)  # (B, H, W, P, embed_dim)
        output = self.output_proj(concat)      # (B, H, W, P, 5)
        return output

class TransformerFFN(Layer):
    def __init__(self, hidden_dim, output_dim):
        super(TransformerFFN, self).__init__()
        #self.dense1 = Dense(hidden_dim, activation='relu',kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)
        self.dense1 = Dense(hidden_dim,kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)
        self.dense2 = Dense(output_dim,kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)

    def call(self, inputs):
        # inputs: (H, W, P, V)
        x = self.dense1(inputs)  # (H, W, P, hidden_dim)
        x = gelu_approx(x)
        x = self.dense2(x)       # (H, W, P, output_dim)
        return x



class TransSingleUnit(Layer):
    def __init__(self, input_dim, embed_dim, num_heads, ffn_hidden_dim):
        super(TransSingleUnit, self).__init__()
        self.mha= MultiHeadAttention(input_dim=input_dim,num_heads=num_heads)
        self.ffn=TransformerFFN(hidden_dim=ffn_hidden_dim,output_dim=input_dim)
        self.lnm1=LayerNormalization(epsilon=1e-6)
        self.lnm2=LayerNormalization(epsilon=1e-6)
    def call(self,X):
        #pre-LN
        # mha_output=self.mha(self.lnm1(X))
        # X=X+mha_output
        # ##
        # ffn_output=self.ffn(self.lnm2(X))
        # X=X+ffn_output
        # return X

        #Post-LN
        X=self.lnm1(self.mha(X)+X)
        X=self.lnm2(self.ffn(X)+X)
        return X


class GAPLayer(Layer):
    def __init__(self):
        super(GAPLayer,self).__init__()
    def call(self,X,X2):
        X=tf.reduce_mean(X,axis=[1,2],keepdims=True)
        return X

class IFFM(Layer):
    def __init__(self,final_dim):
        super(IFFM, self).__init__()
        self.final_dim=final_dim
        self.lnm=LayerNormalization(epsilon=1e-6)
        self.final_dim_double=self.final_dim*2
        self.linear=LearnableMultiplication(input_dim=self.final_dim_double, output_dim=self.final_dim)
    def call(self,inputs1,inputs2):
        inputs2 = tf.transpose(inputs2, perm=[0, 1, 2, 4, 3])  # (B, H, W,Y,X) to (B, H, W,X,Y)
        x=tf.concat([inputs1,inputs2],axis=-1)
        x=self.linear(x)
        x=gelu_approx(x)
        return x

class SpecConvNext(Layer):
    def __init__(self,kernel_length,input_filters,conv_groups=1):
        super(SpecConvNext,self).__init__()
        self.Conv3D_a=Conv3D(filters=input_filters,kernel_size=(1,1,kernel_length),groups=conv_groups,padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)
        self.Conv3D_b=Conv3D(filters=4*input_filters,kernel_size=(1,1,1),groups=conv_groups,padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)
        self.Conv3D_c=Conv3D(filters=input_filters,kernel_size=(1,1,1),groups=conv_groups,padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2)
        self.lnm=LayerNormalization(epsilon=1e-6)

    def call(self,x):
        x=self.Conv3D_a(x)
        x=self.lnm(x)
        x=self.Conv3D_b(x)
        x=gelu_approx(x)
        x=self.Conv3D_c(x)
        return x





class FESCNPTU_V3(Layer):
    def __init__(self,kernel_length,input_filters,conv_groups,input_dim, embed_dim, num_heads,ffn_hidden_dim):
        super(FESCNPTU_V3,self).__init__()
        self.scn=SpecConvNext(kernel_length=kernel_length,input_filters=input_filters,conv_groups=conv_groups)
        self.ptu=TransSingleUnit(input_dim=input_dim,embed_dim=embed_dim,num_heads=num_heads,ffn_hidden_dim=ffn_hidden_dim)
        self.iffm=IFFM(final_dim=input_dim)
    def call(self,scn_inputs,ptu_inputs):
        scn_op=self.scn(scn_inputs)
        ptu_op=self.ptu(ptu_inputs)
        ptu_op=self.iffm(ptu_op,scn_op)
        scn_op=scn_op+scn_inputs
        ptu_op=ptu_op+ptu_inputs
        return (scn_op,ptu_op)

class ReduceSpatial_v2(Layer):
    def __init__(self,**kwargs):
        super().__init__(**kwargs)
    def call(self, inputs1,inputs2):
        ## Softmax
        # soft_ip2 = tf.exp(inputs2) / tf.reduce_sum(tf.exp(inputs2), axis=[1,2], keepdims=True)
        # tns_mul=inputs1*soft_ip2
        # tf_reduce=tf.reduce_mean(tns_mul, axis=[1,2], keepdims=True)
        ##
        tns_mul=inputs1*inputs2
        tns_mul_sum=tf.reduce_sum(tns_mul, axis=[1,2], keepdims=True)
        tns_wgt_sum=tf.reduce_sum(inputs2, axis=[1,2], keepdims=True)
        tf_reduce=tf.math.divide_no_nan(tns_mul_sum,tns_wgt_sum)
        return tf_reduce
    def get_config(self):
        config = super().get_config()
        return config



def conv3d_output_shape(input_shape, kernel_size, strides=(1,1,1), padding='valid', filters=1):
    H, W, D, _ = input_shape
    k1, k2, k3 = kernel_size
    s1, s2, s3 = strides

    if padding.lower() == 'valid':
        out_H = math.floor((H - k1) / s1) + 1
        out_W = math.floor((W - k2) / s2) + 1
        out_D = math.floor((D - k3) / s3) + 1
    elif padding.lower() == 'same':
        out_H = math.ceil(H / s1)
        out_W = math.ceil(W / s2)
        out_D = math.ceil(D / s3)
    else:
        raise ValueError("padding must be either 'valid' or 'same'")

    return (out_H, out_W, out_D, filters)

class StemTranspose(Layer):
    def __init__(self):
        super(StemTranspose,self).__init__()
    def call(self,x):
        x=tf.transpose(x, perm=[0, 1, 2, 4, 3])
        return x


class SCNChannelMix(Layer):
    def __init__(self,groups):
        super(SCNChannelMix,self).__init__()
        self.groups=groups
    def call(self,x):
        if(self.groups==1):
            return x
        #x: (B, H, W, S, C)
        B, H, W, S, C = tf.unstack(tf.shape(x))
        group_channels=C//self.groups
        x=tf.reshape(x,(B, H, W, S,self.groups,group_channels)) # (B,H,W,S,groups,group_channels)
        x=tf.transpose(x, perm=[0, 1, 2, 3, 5, 4])
        x= tf.reshape(x,(B, H, W, S,self.groups*group_channels)) # (B,H,W,S,groups*group_channels=C)
        return x


In [ ]:


##Model with multiple TU, with only l2 regularization, no using of multi level features
def model_ASCT(input_shapes,stem_filters,stem_kl,stem_strides,scn_kl,scn_groups,num_heads,fe_levels,num_classes,ip_dim_stem,arc_flag=0):
    input1_shape=input_shapes[0]
    input2_shape=input_shapes[1]
    #
    inputs=Input(shape=input1_shape)
    inputs2=Input(shape=input2_shape)
    ## STEM layers
    stem_features=Conv3D(filters=stem_filters[0],kernel_size=(1,1,stem_kl[0]),strides=(1,1,stem_strides[0]),padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2) (inputs)
    stem_features=Conv3D(filters=stem_filters[1],kernel_size=(1,1,stem_kl[1]),strides=(1,1,stem_strides[1]),padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2) (stem_features)
    stem_features=Conv3D(filters=stem_filters[2],kernel_size=(1,1,stem_kl[2]),strides=(1,1,stem_strides[2]),padding="same",
                           kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2) (stem_features)
    #stem_op_shape=conv3d_output_shape(input_shape,(1,1,stem_kl),(1,1,stem_kl), padding='same', filters=stem_filters)
    # stem_Trans=StemTranspose()(stem_features)
    #
    # input_dim=input2_shape[-2]
    input_dim=ip_dim_stem
    embed_dim=input_dim*num_heads
    ffn_hidden_dim=min(4,num_heads)*input_dim
    ##
    for i in range(fe_levels[0]):
        # fe_i=FESCNPTU_V3(kernel_length=scn_kl,input_filters=stem_filters[2],conv_groups=scn_groups,input_dim=input_dim,
        #               embed_dim=embed_dim, num_heads=num_heads,ffn_hidden_dim=ffn_hidden_dim)
        # stem_features,stem_Trans=fe_i(stem_features,stem_Trans)
        scnxt=SpecConvNext(kernel_length=scn_kl,input_filters=stem_filters[2],conv_groups=scn_groups)
        stem_features=scnxt(stem_features)+stem_features
    ##
    stem_Trans=StemTranspose()(stem_features)
    for i in range(fe_levels[1]):
        tsu=TransSingleUnit(input_dim=input_dim,embed_dim=embed_dim,num_heads=num_heads,ffn_hidden_dim=ffn_hidden_dim)
        stem_Trans=tsu(stem_Trans)

    #
    stem_Trans=StemTranspose()(stem_Trans)
    #
    X_CF=stem_features
    if(fe_levels[1]>0 and arc_flag==0):
        X_CF=stem_Trans
    if(fe_levels[1]>0 and arc_flag==1):
        X_CF=Concatenate()([stem_features,stem_Trans])
    #
    # X_CF=Concatenate()([stem_features,stem_Trans])
    X_SF=ReduceSpatial_v2()(X_CF,inputs2) #ReduceSpatial_v2()(X_CF,inputs2) #ReduceSpatial_v2
    X_Flat=Flatten()(X_SF)
    Y=Dense(num_classes,kernel_regularizer=rl_l1,bias_regularizer=rl_l1)(X_Flat) #kernel_regularizer=rl_l1_l2,bias_regularizer=rl_l1_l2
    # Y=PrunableDense(num_classes, activation=None, target_sparsity=0.75, l1_reg=0.001,name='prune_dense_final')(X_Flat)
    Y=Softmax()(Y)
    model=Model(inputs=[inputs,inputs2], outputs=Y)
    adam = Adam(learning_rate=base_learning_rate)
    model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
file_path_temp = '/content/drive/MyDrive/IndianPines/indian_pines_corrected.npy'
temp = np.load(file_path_temp)
# temp=temp[:,:,0:102]
file_path_gt_temp = '/content/drive/MyDrive/IndianPines/indian_pines_gt.npy'
gt_temp=np.load(file_path_gt_temp)
# temp_og=np.copy(temp)

In [ ]:
## Execution Parameters
base_learning_rate=0.0001
total_epochs=1000
batch_size=64
rl_l1_l2=regularizers.L1L2(l1=0,l2=0.001)
rl_l1=regularizers.L1L2(l1=0.001,l2=0)
p_size=9
p_levels=p_size//2+1
dataset_classes=16
##
stem_kl=[3,3,3]
stem_filters=[16,16,16]
stem_strides=[2,2,3]
##
scn_kl=7
scn_groups=1
## Arcitecture flag
arc_flag=0
## Feature extraction Levels
fe_levels=(0,10)
##
trans_mha_heads=7

##
Trn_IP=[0,10,71,41,11,24,36,10,23,10,48,122,29,10,63,19,10]
Vld_IP=[0,10,71,41,11,24,36,10,23,5,48,122,29,10,63,19,10]

# Trn_IP=[0,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20]
# Vld_IP=[0,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20,20]

# Trn_IP=[0,20,20,20,20,20,20,20,20,20]
# Vld_IP=[0,20,20,20,20,20,20,20,20,20]

# Trn_IP=[0,10,10,10,10,10,10,10,10,10,10,10,10,10]
# Vld_IP=[0,10,10,10,10,10,10,10,10,10,10,10,10,10]

In [ ]:
# %%capture cap
total_experiments=5
all_class_matrix=np.zeros((dataset_classes,total_experiments))
all_AOK_matrix=np.zeros((3,total_experiments))
for rsi in range(total_experiments):
    print('-----------------------------------------------------')
    ran_seed=rsi+100
    np.random.seed(ran_seed)
    # tf.keras.utils.set_random_seed(ran_seed)
    # tf.config.experimental.enable_op_determinism() # Ensures deterministic operations
    ##
    X_Train,Y_Train,Y_Tr_hot,X_Test,Y_Test,Y_Ts_hot,X_Vld,Y_Vld,Y_Vld_hot,X_og_Tr,X_og_Vld,X_og_Ts,ITPNO=dataset_preparation(p_size)
    print ('test run successful')
    X_Train_Ex=np.expand_dims(X_Train,axis=-1)
    X_Test_Ex=np.expand_dims(X_Test,axis=-1)
    X_Vld_Ex=np.expand_dims(X_Vld,axis=-1)
    del X_Train,X_Test,X_Vld
    ##
    ip1_shape=X_Train_Ex.shape
    input1_shape=(ip1_shape[-4],ip1_shape[-3],ip1_shape[-2],ip1_shape[-1])
    #
    stem_op_shape=conv3d_output_shape(input1_shape,(1,1,stem_kl[0]),(1,1,stem_strides[0]), padding='same', filters=stem_filters[0])
    stem_op_shape=conv3d_output_shape(stem_op_shape,(1,1,stem_kl[1]),(1,1,stem_strides[1]), padding='same', filters=stem_filters[1])
    stem_op_shape=conv3d_output_shape(stem_op_shape,(1,1,stem_kl[2]),(1,1,stem_strides[2]), padding='same', filters=stem_filters[2])
    #
    # X_og_Tr_Ex=np.tile(np.expand_dims(X_og_Tr,axis=(-2,-1)),(1,1,1,stem_op_shape[-2],(arc_flag+1)*stem_op_shape[-1]) )
    # X_og_Vld_Ex=np.tile(np.expand_dims(X_og_Vld,axis=(-2,-1)),(1,1,1,stem_op_shape[-2],(arc_flag+1)*stem_op_shape[-1]))
    # X_og_Ts_Ex=np.tile(np.expand_dims(X_og_Ts,axis=(-2,-1)),(1,1,1,stem_op_shape[-2],(arc_flag+1)*stem_op_shape[-1]))
    X_og_Tr_Ex=np.expand_dims(X_og_Tr,axis=(-2,-1))
    X_og_Vld_Ex=np.expand_dims(X_og_Vld,axis=(-2,-1))
    X_og_Ts_Ex=np.expand_dims(X_og_Ts,axis=(-2,-1))
    del X_og_Tr,X_og_Vld,X_og_Ts
    print('final train, validation, test dataet shapes')
    print(X_Train_Ex.shape,X_Vld_Ex.shape,X_Test_Ex.shape)
    print(X_og_Tr_Ex.shape,X_og_Vld_Ex.shape,X_og_Ts_Ex.shape)
    #
    ip2_shape=X_og_Tr_Ex.shape
    input2_shape=(ip2_shape[-4],ip2_shape[-3],ip2_shape[-2],ip2_shape[-1])
    # modelXformer(input_shapes,stem_filters,stem_kl,num_heads,fe_levels,num_classes):
    input_shapes=(input1_shape,input2_shape)
    print('final input shape :', input_shapes)
    hsicl=model_ASCT(input_shapes,stem_filters,stem_kl,stem_strides,scn_kl,scn_groups,num_heads=trans_mha_heads,
                          fe_levels=fe_levels,num_classes=dataset_classes,ip_dim_stem=stem_op_shape[-2],arc_flag=arc_flag)
    early_stopping = EarlyStopping(monitor='val_loss',mode='min',patience=10,restore_best_weights=True)
    if(rsi==0):
      print(hsicl.summary())
    # verbose=1 if(rsi==0) else 0
    history=hsicl.fit([X_Train_Ex,X_og_Tr_Ex],Y_Tr_hot,epochs=total_epochs,batch_size=batch_size, verbose=0,
                      validation_data=([X_Vld_Ex,X_og_Vld_Ex],Y_Vld_hot),callbacks=[early_stopping])
    del X_Train_Ex,X_Vld_Ex,X_og_Tr_Ex,X_og_Vld_Ex,Y_Tr_hot,Y_Vld_hot
    Y_pred_hot=hsicl.predict([X_Test_Ex,X_og_Ts_Ex],batch_size=16,verbose=0)
    del X_Test_Ex,X_og_Ts_Ex
    Y_Pred=hot_to_labels(Y_pred_hot)
    print('Metrics for rand seed: ',ran_seed)
    all_class_matrix[:,rsi:rsi+1],all_AOK_matrix[0,rsi],all_AOK_matrix[1,rsi]=cal_class_accuracies(Y_Test,Y_Pred,dataset_classes)
    kappa = cohen_kappa_score(Y_Test,Y_Pred)
    print("Kappa Coefficient: ", kappa)
    all_AOK_matrix[2,rsi]=kappa
    print('No Epochs', len(history.history['loss']))
    #
    # pp_ip=total_hsimage_prediction(p_size,hsicl)
    # file_path_full_image = '/content/drive/MyDrive/IndianPines/ASCT_Full_Image_IP.npy'
    # np.save(file_path_full_image,pp_ip)
    del hsicl,history,Y_pred_hot,Y_Pred



print('********-----------------------------------------------------------------------------------*********')

print('All Statistics')
for cls in range(dataset_classes):
    print(np.mean(all_class_matrix[cls]))

print('mean of AA ',np.mean(all_AOK_matrix[0]))
print('std of AA ',np.std(all_AOK_matrix[0]))

print('mean of OA',np.mean(all_AOK_matrix[1]))
print('std of OA',np.std(all_AOK_matrix[1]))

print('mean of kappa',np.mean(all_AOK_matrix[2]))
print('std of kappa',np.std(all_AOK_matrix[2]))

In [ ]:
import time
import tensorflow as tf
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
from tensorflow.compat.v1 import profiler

def get_flops_keras_v2(model, batch_size=1):
    # Handle single or multiple inputs
    if isinstance(model.input_shape, list):
        input_specs = []
        for shape in model.input_shape:
            shape = [batch_size] + [d if d is not None else 1 for d in shape[1:]]
            input_specs.append(tf.TensorSpec(shape, tf.float32))
    else:
        shape = [batch_size] + [d if d is not None else 1 for d in model.input_shape[1:]]
        input_specs = tf.TensorSpec(shape, tf.float32)

    # Convert model to concrete function
    concrete = tf.function(model).get_concrete_function(input_specs)

    # Freeze and get graph
    frozen_func = convert_variables_to_constants_v2(concrete)
    graph_def = frozen_func.graph.as_graph_def()

    # Profile FLOPs
    with tf.Graph().as_default() as graph:
        tf.graph_util.import_graph_def(graph_def, name="")
        flops = profiler.profile(
            graph,
            options=profiler.ProfileOptionBuilder.float_operation())
    return flops.total_float_ops


In [ ]:
stem_filters=[16,16,16]
trans_mha_heads=7
fe_levels=(10,10)
input_shapes=((9,9,200,1),(9,9,1,1))
batch_samples=256
hsicl=model_ASCT(input_shapes,stem_filters,stem_kl,stem_strides,scn_kl,scn_groups,num_heads=trans_mha_heads,
                          fe_levels=fe_levels,num_classes=dataset_classes,ip_dim_stem=17,arc_flag=arc_flag)
sample_input=np.random.rand(batch_samples,9,9,200,1)
sample_input2=np.random.rand(batch_samples,9,9,1,1)
print(hsicl.summary())
cpu_start=time.process_time()
some_output=hsicl.predict([sample_input,sample_input2],batch_size=batch_samples)
cpu_end=time.process_time()
print('Inference Time ',(cpu_end-cpu_start)*1000/batch_samples)
print("FLOPs:", get_flops_keras_v2(hsicl, batch_size=1))
# print("FLOPs:", get_flops(hsicl, batch_size=1))